# 02 — Generate ML Training Data from Quantum Algorithms

**Hero demo:** One quantum algorithm → thousands of labelled ML training examples.

WestQuant Open automatically generates `(state, action, next_state, reward)` tuples by varying compilation configurations. This is the data factory for training WQT20 and other quantum ML models.

**Key idea:** The same algorithm compiled differently produces different resource metrics. Each compilation path is a training example.

In [ ]:
# Install if needed
# !pip install westquant[qiskit] pandas matplotlib

from qiskit import QuantumCircuit
from qiskit.circuit.library import QFTGate
from westquant import generate_training_data, write_wqdf_jsonl
import pandas as pd

print("WestQuant Open — ML Dataset Generation")
print("=" * 50)

## Step 1: Build the Input Circuit

In [ ]:
# Build a 12-qubit QFT circuit — the input algorithm
n_qubits = 12
qc = QuantumCircuit(n_qubits)
qc.append(QFTGate(n_qubits), range(n_qubits))
qc = qc.decompose(reps=3)

print(f"Input: QFT-{n_qubits}")
print(f"  Depth:     {qc.depth()}")
print(f"  2Q gates:  {sum(1 for inst in qc.data if len(inst.qubits) == 2)}")
print(f"  Total gates: {qc.size()}")
print(f"\nThis single circuit will become thousands of ML training examples.")

## Step 2: Generate the Dataset

We vary over:
- **Optimization levels** (0, 1, 2, 3) — Qiskit's transpiler optimization
- **Basis gate sets** — different hardware native gates
- **Layout methods** — how logical qubits map to physical qubits
- **Routing methods** — how to handle non-adjacent two-qubit gates

Each combination produces a different representation with different metrics.

In [ ]:
import time

# Generate a large dataset from this single circuit
print("Generating ML training data...")
start = time.time()

samples = generate_training_data(
    qc,
    framework="qiskit",
    samples=500,
    seed=42,
    optimization_levels=[0, 1, 2, 3],
    basis_gates_options=[
        ["cx", "u3", "u1", "u2"],
        ["cx", "rz", "sx", "x"],
        ["ecr", "rz", "sx", "x"],
        ["cz", "rz", "sx", "x"],
    ],
    layout_methods=["trivial", "dense", "sabre"],
    routing_methods=["sabre", "stochastic", "basic"],
)

elapsed = time.time() - start
print(f"\nGenerated {len(samples)} training examples in {elapsed:.1f}s")
print(f"Throughput: {len(samples)/elapsed:.0f} samples/second")
print(f"\n1 algorithm (QFT-{n_qubits}) -> {len(samples)} ML training examples")

## Step 3: Convert to ML Training Format

Each sample is a `(state, action, next_state, reward)` tuple:
- **state**: circuit metrics before compilation (depth, 2Q gates, size)
- **action**: compilation config (opt level, basis gates, layout, routing)
- **next_state**: circuit metrics after compilation
- **reward**: improvement in metrics (negative delta = better)

In [ ]:
# Convert to DataFrame
records = []
baseline_depth = qc.depth()
baseline_2q = sum(1 for inst in qc.data if len(inst.qubits) == 2)
baseline_size = qc.size()

for s in samples:
    records.append({
        "sample_id": s.sample_id,
        "representation": s.representation,
        "framework": s.framework,
        "backend": s.backend,
        "num_qubits": s.num_qubits,
        # State (before)
        "baseline_depth": baseline_depth,
        "baseline_2q": baseline_2q,
        "baseline_size": baseline_size,
        # Next state (after compilation)
        "compiled_depth": s.depth,
        "compiled_2q": s.two_qubit_gates,
        "compiled_size": s.size,
        "compiled_swaps": s.swap_gates,
        # Reward (negative delta = improvement)
        "delta_depth": s.depth - baseline_depth,
        "delta_2q": s.two_qubit_gates - baseline_2q,
        "delta_size": s.size - baseline_size,
    })

df = pd.DataFrame(records)
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head(10)

## Step 4: Save in WQDF Format

WestQuant Data Format (WQDF) is the standardized JSONL schema for training data.

In [ ]:
# Save in WQDF JSONL format
output_path = "qft12_training_data.jsonl"
n_written = write_wqdf_jsonl(samples, output_path)
print(f"Saved {n_written} samples to {output_path}")

# Also save as CSV for easy analysis
df.to_csv("qft12_training_data.csv", index=False)
print(f"Saved CSV: qft12_training_data.csv")

# Show schema version
print(f"\nWQDF schema version: {samples[0].schema_version}")
print(f"Total records: {len(samples)}")

## Step 5: Analyze the Dataset

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Depth distribution
axes[0, 0].hist(df["compiled_depth"], bins=30, edgecolor="black", color="steelblue")
axes[0, 0].set_title("Compiled Depth Distribution")
axes[0, 0].set_xlabel("Depth")
axes[0, 0].axvline(baseline_depth, color="red", linestyle="--", label=f"Baseline: {baseline_depth}")
axes[0, 0].legend()

# 2Q gate distribution
axes[0, 1].hist(df["compiled_2q"], bins=30, edgecolor="black", color="coral")
axes[0, 1].set_title("Compiled 2Q Gate Distribution")
axes[0, 1].set_xlabel("2-Qubit Gates")
axes[0, 1].axvline(baseline_2q, color="red", linestyle="--", label=f"Baseline: {baseline_2q}")
axes[0, 1].legend()

# Delta depth (reward)
axes[1, 0].hist(df["delta_depth"], bins=30, edgecolor="black", color="green", alpha=0.7)
axes[1, 0].set_title("Depth Change (Reward)")
axes[1, 0].set_xlabel("Δ Depth (negative = improvement)")
axes[1, 0].axvline(0, color="black", linestyle="-", linewidth=0.5)

# Delta 2Q (reward)
axes[1, 1].hist(df["delta_2q"], bins=30, edgecolor="black", color="purple", alpha=0.7)
axes[1, 1].set_title("2Q Gate Change (Reward)")
axes[1, 1].set_xlabel("Δ 2Q Gates (negative = improvement)")
axes[1, 1].axvline(0, color="black", linestyle="-", linewidth=0.5)

plt.suptitle(f"QFT-{n_qubits}: {len(samples)} ML Training Examples from 1 Algorithm", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("ml_dataset_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 6: Scale Up — Multiple Algorithms

The real power comes from running this across many algorithm families.

In [ ]:
from qiskit.circuit.library import QFTGate, GroverOperator
import numpy as np

# Define multiple algorithm families
def make_qft(n):
    qc = QuantumCircuit(n)
    qc.append(QFTGate(n), range(n))
    return qc.decompose(reps=3)

def make_ghz(n):
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)
    return qc

def make_grover(n, iterations=1):
    """Simple Grover-like circuit."""
    qc = QuantumCircuit(n)
    qc.h(range(n))
    for _ in range(iterations):
        qc.append(GroverOperator(n), range(n))
    return qc.decompose(reps=2)

# Generate datasets for each family
families = {
    "GHZ": [make_ghz(n) for n in [4, 6, 8, 10, 12]],
    "QFT": [make_qft(n) for n in [4, 6, 8, 10, 12]],
    "Grover": [make_grover(n) for n in [4, 6, 8]],
}

all_samples = []
for family_name, circuits in families.items():
    for qc in circuits:
        samples = generate_training_data(
            qc, framework="qiskit", samples=100, seed=42,
            optimization_levels=[0, 1, 2, 3],
        )
        all_samples.extend(samples)
        print(f"  {family_name}-{qc.num_qubits}: {len(samples)} samples")

print(f"\nTotal: {len(all_samples)} training examples from {sum(len(v) for v in families.values())} circuits")
print(f"Algorithm families: {list(families.keys())}")

## Summary

| Input | Output |
|-------|--------|
| 1 algorithm (QFT-12) | 500 ML training examples |
| 13 circuits (3 families) | 1,300+ ML training examples |
| 10 algorithm families × 20-50 instances | ~10,000+ training examples |

Each example is a `(state, action, next_state, reward)` tuple ready for reinforcement learning.

**This is the data factory.** WQT20 and future models train on this data — not on synthetic datasets, but on real compilation trajectories from real quantum algorithms.